<a href="https://colab.research.google.com/github/dhana-l-tech/examsphere/blob/main/Final_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

comparison = pd.DataFrame({

"Metric":[
"Dataset Size",
"Features",
"Classes",
"Accuracy",
"Precision",
"Recall",
"F1 Score",
"Average Confidence",
"Average Entropy",
"Explanation Reduction (%)",
"Training Time (sec)"
],

"ML-EdgeIIoT":[
152429,
58,
15,
0,   # Fill later
0,
0,
0,
0,
0,
0,
0
],

"UNSW-NB15":[
257673,
34,
10,
0.8180,
0.8113,
0.8180,
0.8051,
0.8136,
0,      # Fill entropy
79.24,
123.77
]

})

comparison

,Metric,ML-EdgeIIoT,UNSW-NB15
0,Dataset Size,152429,257673.0000
1,Features,58,34.0000
2,Classes,15,10.0000
3,Accuracy,0,0.8180
4,Precision,0,0.8113
5,Recall,0,0.8180
6,F1 Score,0,0.8051
7,Average Confidence,0,0.8136
8,Average Entropy,0,0.0000
9,Explanation Reduction (%),0,79.2400


In [2]:
print("rf" in globals())

False


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("✅ Random Forest model recreated successfully!")

NameError: name 'X_train' is not defined

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import pandas as pd

df = pd.read_csv("/content/edgeiiot_cleaned (1).csv")

print("="*60)
print("Dataset Loaded Successfully")
print("="*60)

print("Shape :", df.shape)
print("\nColumns:\n")
print(df.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(["Attack_label", "Attack_type"], axis=1)
y = df["Attack_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("="*60)
print("TRAIN TEST SPLIT COMPLETED")
print("="*60)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

print("="*60)
print("TRAINING RANDOM FOREST")
print("="*60)

start = time.time()

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

end = time.time()

print("\nModel Trained Successfully!")
print("Training Time:", round(end - start, 2), "seconds")

In [ ]:
# Select one attack sample

attack_sample = X_test.iloc[[0]]

prediction = rf.predict(attack_sample)[0]
confidence = rf.predict_proba(attack_sample).max()

print("Predicted Attack Class :", prediction)
print("Confidence :", round(confidence, 3))

In [ ]:
# Edge-IIoT Severity Mapping
severity_map = {
    0:0,
    1:1,
    2:2,
    3:3,
    4:3,
    5:3,
    6:3,
    7:3,
    8:2,
    9:2,
    10:3,
    11:3,
    12:3,
    13:3,
    14:3
}

severity = severity_map[prediction]

print("Severity :", severity)

# Context-Aware Policy
if confidence < 0.70:
    print("Reason : Low Confidence")
    print("🚨 Generate Explanation")

elif severity >= 2:
    print("Reason : High Severity Attack")
    print("🚨 Generate Explanation")

else:
    print("Reason : Low Risk & High Confidence")
    print("✅ Explanation Suppressed")

In [ ]:
# Find the first sample that should NOT require an explanation

for i in range(len(X_test)):
    sample = X_test.iloc[[i]]

    pred = rf.predict(sample)[0]
    conf = rf.predict_proba(sample).max()

    severity = severity_map[pred]

    if conf >= 0.70 and severity < 2:
        print("Sample Index:", i)
        print("Predicted Class:", pred)
        print("Confidence:", round(conf,3))
        print("Severity:", severity)
        print("✅ This explanation should be SUPPRESSED.")
        break

In [ ]:
if confidence < 0.70:
    print("\n🚨 ALERT")
    print("Attack Detected!")
    print(f"Predicted Attack : {prediction}")
    print(f"Confidence       : {confidence:.2f}")
    print("Reason           : Low confidence prediction")
    print("Action           : Generate SHAP Explanation")

elif severity >= 2:
    print("\n🚨 ALERT")
    print("Attack Detected!")
    print(f"Predicted Attack : {prediction}")
    print(f"Confidence       : {confidence:.2f}")
    print("Reason           : High severity attack")
    print("Action           : Generate SHAP Explanation")

else:
    print("\n✅ ALERT")
    print("Attack Detected!")
    print(f"Predicted Attack : {prediction}")
    print(f"Confidence       : {confidence:.2f}")
    print("Reason           : Low severity & High confidence")
    print("Action           : Explanation Suppressed")